# MeterMind — Model 1: Rule-Based DP Reorderer

This notebook implements the **rule-based baseline** for MeterMind.

Given an expanded archaic prose line (e.g. `Thou feedest thy light's own flame with fuel that is self-substantial`), the DP reorderer finds the word ordering that **best matches the iambic pentameter stress pattern** — purely through symbolic constraint-solving, no learning involved.

### Why DP and not greedy?
A greedy approach fills each stress position with the first matching word it finds. This can paint itself into a corner — grabbing a word early that would have been more useful later. Dynamic programming instead considers **all possible orderings** and finds the globally optimal one.

### Pipeline
```
training_pairs.csv → stress extraction (CMU) → DP reorder → evaluate (MA, SP, G)
```

## 0. Install dependencies

In [ ]:
%pip install pronouncing sentence-transformers openai --quiet

## 1. Imports

In [ ]:
import re
import csv
import io
import math
import pronouncing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from itertools import permutations
from sentence_transformers import SentenceTransformer, util

print('Imports ready.')

## 2. Load dataset

In [ ]:
# Upload training_pairs.csv when prompted
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))

# Column names from dataset_generation.ipynb are 'input' and 'target'
pairs = list(zip(df['input'].tolist(), df['target'].tolist()))

print(f'Loaded {len(pairs):,} pairs')
print(f'\nExample:')
print(f'  INPUT  : {pairs[0][0]}')
print(f'  TARGET : {pairs[0][1]}')

## 3. Stress extraction

Maps each word to its syllable stress pattern using the CMU Pronouncing Dictionary.

- **Multisyllabic words** (e.g. `compare` → `[0,1]`) have a fixed stress pattern.
- **Monosyllabic words marked as stressed** (e.g. `shall`, `thee`, `day`) are treated as **flexible** — they can sit in either a stressed or unstressed position. This reflects how poets naturally use function words.

In [ ]:
def tokenize(text):
    """Lowercase, strip punctuation, split into word tokens."""
    return re.sub(r"[^a-zA-Z\s]", "", text.lower()).split()


def get_stress(word):
    """
    Return the syllable stress sequence for a word as a list of ints.
    1 = stressed, 0 = unstressed.
    CMU uses 0/1/2 where 2 is secondary stress — we treat that as 1.

    Fallback for archaic suffixes not in CMU:
    - *est (feedest, makest) → strip suffix, look up root, append [0]
    - *eth (feedeth, maketh) → strip suffix, look up root, append [0]
    Both suffixes are always unstressed so appending [0] is correct.
    If root also not found, default to [0].
    """
    phones = pronouncing.phones_for_word(word)
    if not phones:
        # Try stripping archaic verb suffixes and looking up the root
        for suffix in ['est', 'eth']:
            if word.endswith(suffix) and len(word) > len(suffix) + 1:
                root = word[:-len(suffix)]
                root_phones = pronouncing.phones_for_word(root)
                if root_phones:
                    stress = [1 if s == '2' else int(s)
                              for s in pronouncing.stresses(root_phones[0]) if s in '012']
                    return stress + [0]  # suffix syllable is always unstressed
        return [0]  # genuine unknown
    stress = [1 if s == '2' else int(s)
              for s in pronouncing.stresses(phones[0]) if s in '012']
    # monosyllabic stressed word → flexible: return [0,1] to signal either position ok
    if len(stress) == 1 and stress[0] == 1:
        return [0, 1]
    return stress


def n_syllables(word):
    """Count syllables via CMU phone count. Default 1 if not found."""
    phones = pronouncing.phones_for_word(word)
    if not phones:
        return 1
    return len([p for p in phones[0].split() if p[-1].isdigit()])


def is_flexible(word):
    """True if word is monosyllabic and can fill either stressed or unstressed position."""
    return get_stress(word) == [0, 1]


# Sanity check
test_words = ['shall', 'i', 'compare', 'thee', 'to', 'a', 'summers', 'day', 'feedest', 'flame']
print(f"{'word':12} syllables  stress      flexible")
print("-" * 45)
for w in test_words:
    print(f"{w:12} {n_syllables(w):<10} {str(get_stress(w)):12} {is_flexible(w)}")

## 4. DP Reorderer

### How it works

Iambic pentameter has 10 syllable positions: `[0,1,0,1,0,1,0,1,0,1]` (unstressed, stressed, alternating).

The DP solves this as an **assignment problem**:
- **State:** `(syllable_position, set_of_words_used_so_far)`
- **Transition:** try placing each unused word next; score how well its stress pattern matches the template at the current position
- **Goal:** find the word ordering that maximises total stress matches

Because the number of words per line is small (~10-20), we use **bitmask DP** — each word is represented as a bit in an integer, making state representation compact and fast.

### Scoring
Each syllable earns 1 point if it matches the iambic template. Flexible monosyllables earn 1 point in any position. The DP maximises total points across all 10 template positions.

In [ ]:
IAMBIC_TEMPLATE = [0, 1] * 10   # 10 positions: unstressed-STRESSED x5
MAX_TEMPLATE_LEN = len(IAMBIC_TEMPLATE)


def word_syllable_score(word, start_pos):
    """
    Score how well `word` fits into the iambic template starting at `start_pos`.
    Returns (score, syllables_consumed).
    """
    stress = get_stress(word)
    n_syl  = len(stress) if len(stress) > 1 else n_syllables(word)
    score  = 0

    if is_flexible(word):
        # Flexible monosyllable: always scores 1
        score = 1
        syllables_consumed = 1
    else:
        syllables_consumed = 0
        for i, s in enumerate(stress):
            pos = start_pos + i
            if pos < MAX_TEMPLATE_LEN:
                if s == IAMBIC_TEMPLATE[pos]:
                    score += 1
                syllables_consumed += 1
            else:
                break

    return score, max(syllables_consumed, 1)


def dp_reorder(words):
    """
    Find the word ordering that maximises iambic stress match using bitmask DP.

    State: (syllable_position, bitmask_of_used_words)
    Value: best score achievable from this state

    Returns the reordered word list.
    """
    # Cap at 20 words to keep DP tractable (2^20 = 1M states)
    # Words beyond 20 are appended at the end unchanged
    cap      = min(len(words), 20)
    core     = words[:cap]
    leftover = words[cap:]
    n        = len(core)

    # Pre-compute scores for each (word, start_position) pair
    # score_cache[i][pos] = (score, syllables_consumed) for word i at template position pos
    score_cache = {}
    for i, w in enumerate(core):
        for pos in range(MAX_TEMPLATE_LEN):
            score_cache[(i, pos)] = word_syllable_score(w, pos)

    # DP table: dp[mask][pos] = best score with `mask` words used, at syllable position `pos`
    FULL_MASK = (1 << n) - 1
    NEG_INF   = float('-inf')

    # Use dicts for sparse storage
    dp     = {}   # (mask, pos) -> best_score
    parent = {}   # (mask, pos) -> (prev_mask, prev_pos, word_idx)

    dp[(0, 0)] = 0

    # Process states in order of number of bits set (words placed)
    for num_placed in range(n):
        for (mask, pos), score in list(dp.items()):
            if bin(mask).count('1') != num_placed:
                continue
            if pos >= MAX_TEMPLATE_LEN:
                continue

            # Try placing each unused word
            for i in range(n):
                if mask & (1 << i):
                    continue   # already used

                word_score, syl_consumed = score_cache[(i, pos)]
                new_mask  = mask | (1 << i)
                new_pos   = min(pos + syl_consumed, MAX_TEMPLATE_LEN)
                new_score = score + word_score

                if dp.get((new_mask, new_pos), NEG_INF) < new_score:
                    dp[(new_mask, new_pos)]     = new_score
                    parent[(new_mask, new_pos)] = (mask, pos, i)

    # Find best final state (all words placed)
    best_score = NEG_INF
    best_state = None
    for pos in range(MAX_TEMPLATE_LEN + 1):
        s = dp.get((FULL_MASK, pos), NEG_INF)
        if s > best_score:
            best_score = s
            best_state = (FULL_MASK, pos)

    # Backtrack to recover word order
    if best_state is None or best_state not in parent:
        return words   # fallback: return original order

    order = []
    state = best_state
    while state in parent:
        prev_mask, prev_pos, word_idx = parent[state]
        order.append(word_idx)
        state = (prev_mask, prev_pos)
    order.reverse()

    reordered = [core[i] for i in order] + leftover
    return reordered


# Quick test
test_line = "thou feedest thy light own flame with fuel that is self substantial"
words     = tokenize(test_line)
result    = dp_reorder(words)
print('Input    :', ' '.join(words))
print('Reordered:', ' '.join(result))

## 5. Evaluation metrics

Three metrics, matching the MeterMind evaluation framework:

| Metric | What it measures | How |
|--------|-----------------|-----|
| **MA** — Metrical Accuracy | How well the output matches iambic stress pattern | CMU stress vs template |
| **SP** — Semantic Preservation | How much meaning is retained | Cosine similarity of SentenceTransformer embeddings |
| **G** — Grammaticality | How fluent/natural the output reads | GPT-2 perplexity (lower = better) |

In [ ]:
# ── Metrical Accuracy ─────────────────────────────────────────────────────────
def metrical_accuracy(words, template=IAMBIC_TEMPLATE):
    """
    Fraction of template syllable positions correctly filled.
    Flexible monosyllables score 1 in any position.
    """
    correct   = 0
    syl_pos   = 0

    for word in words:
        if syl_pos >= len(template):
            break
        stress = get_stress(word)
        n_syl  = n_syllables(word)

        if is_flexible(word):
            correct += 1
            syl_pos += 1
        else:
            for s in stress:
                if syl_pos >= len(template):
                    break
                if s == template[syl_pos]:
                    correct += 1
                syl_pos += 1

    return correct / len(template)


# ── Semantic Preservation ─────────────────────────────────────────────────────
print('Loading SentenceTransformer (first run downloads ~90MB)...')
sp_model = SentenceTransformer('all-MiniLM-L6-v2')

def semantic_preservation(original, output):
    """Cosine similarity between sentence embeddings. Range [0,1], higher = better."""
    emb = sp_model.encode([original, output], convert_to_tensor=True)
    return float(util.cos_sim(emb[0], emb[1]))


# ── Grammaticality via GPT-2 perplexity ───────────────────────────────────────
# Lower perplexity = more grammatical/fluent
# We normalise to [0,1] as G = 1 / (1 + log(perplexity)) for comparability
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

print('Loading GPT-2 (first run downloads ~500MB)...')
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
gpt2_model     = GPT2LMHeadModel.from_pretrained('gpt2')
gpt2_model.eval()

def grammaticality(text):
    """
    Returns normalised grammaticality score in [0,1].
    G = 1 / (1 + log(perplexity)).
    """
    inputs = gpt2_tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        loss = gpt2_model(**inputs, labels=inputs['input_ids']).loss
    perplexity = math.exp(loss.item())
    return 1 / (1 + math.log(perplexity))


print('All metrics ready.')

## 6. Run DP reorderer on full dataset

In [ ]:
from tqdm.notebook import tqdm

results = []

for prose_input, target in tqdm(pairs, desc='Reordering'):
    words    = tokenize(prose_input)
    reordered = dp_reorder(words)
    output   = ' '.join(reordered)

    ma = metrical_accuracy(reordered)
    sp = semantic_preservation(prose_input, output)
    g  = grammaticality(output)

    results.append({
        'input':    prose_input,
        'target':   target,
        'output':   output,
        'MA':       round(ma, 4),
        'SP':       round(sp, 4),
        'G':        round(g,  4),
    })

results_df = pd.DataFrame(results)
print(f'Done. {len(results_df):,} results.')

## 7. Results

In [ ]:
print('=== DP Reorderer — Aggregate Results ===')
print(f"Metrical Accuracy (MA) : {results_df['MA'].mean():.4f} ± {results_df['MA'].std():.4f}")
print(f"Semantic Preservation (SP): {results_df['SP'].mean():.4f} ± {results_df['SP'].std():.4f}")
print(f"Grammaticality (G)     : {results_df['G'].mean():.4f} ± {results_df['G'].std():.4f}")

print('\n=== Sample outputs ===')
for _, row in results_df.head(5).iterrows():
    print(f"INPUT  : {row['input']}")
    print(f"TARGET : {row['target']}")
    print(f"OUTPUT : {row['output']}")
    print(f"MA={row['MA']:.3f}  SP={row['SP']:.3f}  G={row['G']:.3f}")
    print()

In [ ]:
# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('DP Reorderer — Metric Distributions', fontsize=14)

for ax, metric, colour in zip(axes, ['MA', 'SP', 'G'], ['steelblue', 'seagreen', 'tomato']):
    ax.hist(results_df[metric], bins=30, color=colour, edgecolor='white', alpha=0.85)
    ax.axvline(results_df[metric].mean(), color='black', linestyle='--', linewidth=1.5, label=f"mean={results_df[metric].mean():.3f}")
    ax.set_title(metric)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.savefig('dp_reorderer_metrics.png', dpi=150)
plt.show()

## 8. Save results

In [ ]:
results_df.to_csv('dp_reorderer_results.csv', index=False)
print('Saved dp_reorderer_results.csv')

# Summary dict — import this into the evaluation notebook to compare all 3 models
import json
summary = {
    'model': 'DP Reorderer',
    'n':     len(results_df),
    'MA':    {'mean': round(float(results_df['MA'].mean()), 4), 'std': round(float(results_df['MA'].std()), 4)},
    'SP':    {'mean': round(float(results_df['SP'].mean()), 4), 'std': round(float(results_df['SP'].std()), 4)},
    'G':     {'mean': round(float(results_df['G'].mean()),  4), 'std': round(float(results_df['G'].std()),  4)},
}
with open('dp_reorderer_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved dp_reorderer_summary.json')

# Download both
files.download('dp_reorderer_results.csv')
files.download('dp_reorderer_summary.json')
files.download('dp_reorderer_metrics.png')